# VisionRAG: Multimodal Retrieval Augmented Generation

**A production-style system for answering questions from PDFs containing text, tables, images, and charts.**

---

## Notebook Cell Map

| Cell | Title | Purpose |
|------|-------|---------|
| Cell 1 | Install Dependencies | Install all required libraries |
| Cell 2 | Import Libraries | Import packages and configure logging |
| Cell 3 | Configure Gemini API | Set API key and initialize Gemini client |
| Cell 4 | Project Folder Structure | Create local directory tree |
| Cell 5 | Relational Schema | Define SQLite schema for documents and chunks |
| Cell 6 | Load / Generate Example PDFs | Download or create synthetic PDFs for testing |
| Cell 7 | Document Parsing Pipeline | Extract text, tables, and images from PDFs |
| Cell 8 | OCR Pipeline | Run PaddleOCR on extracted images |
| Cell 9 | Chunking Strategy | Split text into semantic chunks |
| Cell 10 | Text Embedding Pipeline | Embed text chunks with SentenceTransformers |
| Cell 11 | Image Embedding Pipeline | Embed images with CLIP |
| Cell 12 | Setup Qdrant Vector Database | Initialize local Qdrant collections |
| Cell 13 | Store Embeddings | Upsert all embeddings into Qdrant |
| Cell 14 | Hybrid Retrieval System | Query text and image collections |
| Cell 15 | Cross-Modal Reranker | Combine and rerank multi-modal results |
| Cell 16 | RAG Reasoning with Gemini | Send retrieved context to Gemini for answer |
| Cell 17 | End-to-End Question Answering Pipeline | Assemble full query -> answer pipeline |
| Cell 18 | Evaluation Tests | Run example queries and inspect results |
| Cell 19 | FastAPI Endpoint Example | Production-ready POST /query endpoint |
| Cell 20 | Performance Improvements | Discussion of batch embedding, indexing, caching |
| Cell 21 | Production Deployment Guide | Docker, Next.js frontend, and portfolio tips |

---

## Recommended Public Datasets

You can test VisionRAG with any of the following real-world PDFs:

| Dataset | URL | Content |
|---------|-----|---------|
| Tesla Annual Report 2023 | https://digitalassets.tesla.com/tesla-contents/image/upload/IR/TSLA-Q4-2023-Update.pdf | Financial tables, charts, text |
| NASA Technical Report | https://ntrs.nasa.gov/api/citations/20205003291/downloads/TM-20205003291.pdf | Diagrams, text, tables |
| WHO COVID-19 Report | https://www.who.int/docs/default-source/coronaviruse/situation-reports/20200423-sitrep-94-covid-19.pdf | Charts, text, tables |
| Attention Is All You Need (ArXiv) | https://arxiv.org/pdf/1706.03762.pdf | Academic diagrams, equations, text |

> **Recommended for this notebook:** Download the Attention Is All You Need paper:  
> `wget -O data/raw/attention_paper.pdf https://arxiv.org/pdf/1706.03762.pdf`

The notebook also generates a **synthetic PDF** automatically so you can run it without any download.


## Cell 1 — Install Dependencies

Install all libraries needed for document parsing, embeddings, vector storage, and the LLM API.
This may take a few minutes on first run.

In [ ]:
# Core document processing
!pip install pymupdf unstructured[all-docs] paddleocr paddlepaddle --quiet

# Embeddings
!pip install sentence-transformers transformers torch torchvision --quiet
!pip install open-clip-torch --quiet

# Vector database
!pip install qdrant-client --quiet

# LLM API
!pip install google-generativeai --quiet

# PDF generation (for synthetic dataset)
!pip install reportlab matplotlib pillow --quiet

# Database and API
!pip install fastapi uvicorn nest-asyncio --quiet

print("All dependencies installed.")

## Cell 2 — Import Libraries

Import all required packages and configure logging so we can track what is happening at each stage.

In [ ]:
import os
import json
import uuid
import sqlite3
import logging
import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import fitz  # PyMuPDF
from PIL import Image
import io

import torch
from sentence_transformers import SentenceTransformer
import open_clip

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter,
    FieldCondition, MatchValue
)

import google.generativeai as genai

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("VisionRAG")

print("Libraries imported.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Cell 3 — Configure Gemini API

Set your Gemini API key. Replace `XXXXXXXX` with a real key from https://makersuite.google.com/app/apikey  
The model used is `gemini-1.5-flash` which supports both text and image input.

In [ ]:
API_KEY = "XXXXXXXX"  # Replace with your real Gemini API key

genai.configure(api_key=API_KEY)

GEMINI_MODEL_NAME = "gemini-1.5-flash"
gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)

def call_gemini(prompt: str, image_pil: Optional[Image.Image] = None) -> str:
    """Call Gemini with optional image input."""
    try:
        if image_pil:
            response = gemini_model.generate_content([prompt, image_pil])
        else:
            response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        logger.warning(f"Gemini API call failed: {e}")
        return f"[Gemini unavailable: {e}]"

# Quick test
test_response = call_gemini("Say hello in one sentence.")
print("Gemini test response:", test_response)

## Cell 4 — Project Folder Structure

Create the local directory tree that simulates a production storage backend (like S3).  
All extracted assets will be organized by document ID.

In [ ]:
BASE_DIR = Path("visionrag_data")

DIRS = [
    BASE_DIR / "raw",           # Original uploaded PDFs
    BASE_DIR / "extracted" / "text",
    BASE_DIR / "extracted" / "tables",
    BASE_DIR / "extracted" / "images",
    BASE_DIR / "ocr",           # OCR output text
    BASE_DIR / "chunks",        # Chunked text JSON files
    BASE_DIR / "embeddings",    # Cached numpy embeddings
    BASE_DIR / "db",            # SQLite database
    BASE_DIR / "qdrant_storage",# Local Qdrant persistence
]

for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

print("Project folder structure:")
for d in DIRS:
    print(f"  {d}")

## Cell 5 — Relational Schema (SQLite)

Define a simple relational database to track documents and their chunks.  
This mirrors a production Postgres schema.

```
documents
  id | file_name | upload_date

chunks
  id | doc_id | modality | text | page_number | bounding_box | vector_id
```

Modality values: `text`, `table`, `image`

In [ ]:
DB_PATH = BASE_DIR / "db" / "visionrag.db"

def get_db_connection():
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.executescript("""
        CREATE TABLE IF NOT EXISTS documents (
            id TEXT PRIMARY KEY,
            file_name TEXT NOT NULL,
            upload_date TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS chunks (
            id TEXT PRIMARY KEY,
            doc_id TEXT NOT NULL,
            modality TEXT NOT NULL CHECK(modality IN ('text', 'table', 'image')),
            text TEXT,
            page_number INTEGER,
            bounding_box TEXT,
            vector_id TEXT,
            FOREIGN KEY (doc_id) REFERENCES documents(id)
        );
    """)
    conn.commit()
    conn.close()
    print("Database initialized.")

def register_document(file_name: str) -> str:
    doc_id = str(uuid.uuid4())
    conn = get_db_connection()
    conn.execute(
        "INSERT INTO documents (id, file_name, upload_date) VALUES (?, ?, ?)",
        (doc_id, file_name, datetime.datetime.utcnow().isoformat())
    )
    conn.commit()
    conn.close()
    return doc_id

def register_chunk(
    doc_id: str,
    modality: str,
    text: Optional[str] = None,
    page_number: Optional[int] = None,
    bounding_box: Optional[dict] = None,
    vector_id: Optional[str] = None
) -> str:
    chunk_id = str(uuid.uuid4())
    conn = get_db_connection()
    conn.execute(
        "INSERT INTO chunks (id, doc_id, modality, text, page_number, bounding_box, vector_id) VALUES (?,?,?,?,?,?,?)",
        (
            chunk_id, doc_id, modality, text, page_number,
            json.dumps(bounding_box) if bounding_box else None,
            vector_id
        )
    )
    conn.commit()
    conn.close()
    return chunk_id

init_db()

# Test: inspect schema
conn = get_db_connection()
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
conn.close()
print("Tables:", [t["name"] for t in tables])

## Cell 6 — Load or Generate Example PDF Documents

Two approaches:
1. If you have a real PDF, place it in `visionrag_data/raw/` and update `PDF_PATH`.
2. The cell below **automatically generates a synthetic PDF** containing text, a table (as text), and an embedded chart image.

**Real PDF suggestion:**  
Download the Transformer paper:
```
wget -O visionrag_data/raw/attention_paper.pdf https://arxiv.org/pdf/1706.03762.pdf
```

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch

def create_chart_image(output_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    quarters = ["Q1", "Q2", "Q3", "Q4"]
    revenue = [120, 145, 170, 210]
    axes[0].bar(quarters, revenue, color="steelblue")
    axes[0].set_title("Annual Revenue by Quarter ($M)")
    axes[0].set_ylabel("Revenue ($M)")

    categories = ["Product A", "Product B", "Product C", "Services"]
    shares = [35, 25, 20, 20]
    axes[1].pie(shares, labels=categories, autopct="%1.0f%%", startangle=90)
    axes[1].set_title("Revenue Share by Product")

    plt.tight_layout()
    plt.savefig(output_path, dpi=100, bbox_inches="tight")
    plt.close()

def generate_synthetic_pdf(output_path: str):
    chart_path = str(BASE_DIR / "raw" / "temp_chart.png")
    create_chart_image(chart_path)

    doc = SimpleDocTemplate(output_path, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("VisionRAG Sample Financial Report 2024", styles["Title"]))
    story.append(Spacer(1, 0.3 * inch))

    body = (
        "This report presents the financial performance of Acme Corp for fiscal year 2024. "
        "The company achieved record revenue of $645 million, a 28% increase year-over-year. "
        "Operating margins improved from 12% to 17% driven by cost optimization in the supply chain. "
        "Product A remains the strongest revenue contributor at 35% of total sales. "
        "The engineering team deployed three major platform upgrades in H2 2024, "
        "resulting in a 40% reduction in customer churn. "
        "Looking ahead, Q1 2025 pipeline stands at $180 million with 62 enterprise deals in progress."
    )
    story.append(Paragraph(body, styles["BodyText"]))
    story.append(Spacer(1, 0.3 * inch))

    story.append(Paragraph("Key Financial Metrics", styles["Heading2"]))
    story.append(Spacer(1, 0.15 * inch))

    table_data = [
        ["Metric", "2023", "2024", "YoY Change"],
        ["Total Revenue ($M)", "503", "645", "+28%"],
        ["Gross Margin", "58%", "63%", "+5pp"],
        ["Operating Margin", "12%", "17%", "+5pp"],
        ["Net Income ($M)", "60", "109", "+82%"],
        ["R&D Spend ($M)", "75", "97", "+29%"],
        ["Headcount", "2,100", "2,650", "+26%"],
    ]
    tbl = Table(table_data, colWidths=[2.2*inch, 1.2*inch, 1.2*inch, 1.3*inch])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2E75B6")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#EAF0FA")]),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
    ]))
    story.append(tbl)
    story.append(Spacer(1, 0.4 * inch))

    story.append(Paragraph("Revenue Charts", styles["Heading2"]))
    story.append(Spacer(1, 0.15 * inch))
    story.append(RLImage(chart_path, width=5.5*inch, height=2.2*inch))
    story.append(Spacer(1, 0.2 * inch))
    story.append(Paragraph(
        "Figure 1: Left chart shows quarterly revenue growth in 2024. "
        "Right chart shows revenue distribution by product line.",
        styles["Italic"]
    ))

    doc.build(story)
    print(f"Synthetic PDF saved to: {output_path}")
    if Path(chart_path).exists():
        os.remove(chart_path)

SYNTHETIC_PDF_PATH = str(BASE_DIR / "raw" / "acme_financial_report_2024.pdf")
generate_synthetic_pdf(SYNTHETIC_PDF_PATH)

# You can also point to a downloaded real PDF:
# PDF_PATH = str(BASE_DIR / "raw" / "attention_paper.pdf")
PDF_PATH = SYNTHETIC_PDF_PATH

print(f"Using PDF: {PDF_PATH}")
print(f"File size: {Path(PDF_PATH).stat().st_size / 1024:.1f} KB")

## Cell 7 — Document Parsing Pipeline

Parse the PDF using PyMuPDF (fitz) to extract:
- **Text** per page
- **Images** (saved as PNG files)
- **Tables** detected via word-block bounding boxes (lightweight heuristic; swap for Unstructured.io for production)

Each extracted element gets a `page_number` and `bounding_box` for vector metadata.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ParsedElement:
    modality: str          # 'text', 'image', 'table'
    content: str           # text content or image file path
    page_number: int
    bounding_box: Optional[dict] = None
    image_pil: Optional[Any] = field(default=None, repr=False)


def extract_text_blocks(page, page_number: int) -> List[ParsedElement]:
    elements = []
    blocks = page.get_text("blocks")
    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block
        if block_type == 0 and text.strip():  # type 0 = text
            elements.append(ParsedElement(
                modality="text",
                content=text.strip(),
                page_number=page_number,
                bounding_box={"x0": x0, "y0": y0, "x1": x1, "y1": y1}
            ))
    return elements


def extract_images(page, page_number: int, doc, doc_id: str) -> List[ParsedElement]:
    elements = []
    image_list = page.get_images(full=True)
    for img_index, img_info in enumerate(image_list):
        xref = img_info[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        image_ext = base_image["ext"]
        pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        # Skip very small images (likely icons)
        if pil_image.width < 50 or pil_image.height < 50:
            continue

        save_path = BASE_DIR / "extracted" / "images" / f"{doc_id}_p{page_number}_img{img_index}.png"
        pil_image.save(str(save_path))

        elements.append(ParsedElement(
            modality="image",
            content=str(save_path),
            page_number=page_number,
            image_pil=pil_image
        ))
    return elements


def extract_tables_heuristic(page, page_number: int) -> List[ParsedElement]:
    """
    Lightweight table detection: find dense grids of short text blocks.
    For production use unstructured.io or camelot.
    """
    elements = []
    tables = page.find_tables()
    for table_index, table in enumerate(tables):
        df = table.to_pandas()
        table_text = df.to_string(index=False)
        bbox = table.bbox
        elements.append(ParsedElement(
            modality="table",
            content=table_text,
            page_number=page_number,
            bounding_box={"x0": bbox[0], "y0": bbox[1], "x1": bbox[2], "y1": bbox[3]}
        ))
    return elements


def parse_pdf(pdf_path: str, doc_id: str) -> List[ParsedElement]:
    logger.info(f"Parsing PDF: {pdf_path}")
    all_elements = []
    doc = fitz.open(pdf_path)

    for page_number, page in enumerate(doc, start=1):
        logger.info(f"  Processing page {page_number}/{len(doc)}")
        all_elements.extend(extract_text_blocks(page, page_number))
        all_elements.extend(extract_images(page, page_number, doc, doc_id))
        all_elements.extend(extract_tables_heuristic(page, page_number))

    doc.close()
    logger.info(f"Parsing complete. Total elements: {len(all_elements)}")
    return all_elements


# Register document and parse
doc_id = register_document(Path(PDF_PATH).name)
print(f"Registered document ID: {doc_id}")

parsed_elements = parse_pdf(PDF_PATH, doc_id)

# Summary
modality_counts = {}
for el in parsed_elements:
    modality_counts[el.modality] = modality_counts.get(el.modality, 0) + 1
print("Extracted elements:", modality_counts)

## Cell 8 — OCR Pipeline (PaddleOCR)

Run PaddleOCR on extracted images to get text from charts, scanned diagrams, and screenshots.  
The OCR text is appended to the image element's content so it can also be semantically searched.

In [ ]:
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)

def run_ocr_on_image(image_path: str) -> str:
    result = ocr_engine.ocr(image_path, cls=True)
    if not result or not result[0]:
        return ""
    lines = []
    for line in result[0]:
        text_part = line[1][0]
        confidence = line[1][1]
        if confidence > 0.6:
            lines.append(text_part)
    return " ".join(lines)


def enrich_images_with_ocr(elements: List[ParsedElement]) -> List[ParsedElement]:
    enriched = []
    for el in elements:
        if el.modality == "image" and Path(el.content).exists():
            ocr_text = run_ocr_on_image(el.content)
            if ocr_text:
                el.content = f"[IMAGE_PATH: {el.content}] [OCR_TEXT: {ocr_text}]"
                save_path = BASE_DIR / "ocr" / (Path(el.content).stem + ".txt")
                save_path.write_text(ocr_text, encoding="utf-8")
        enriched.append(el)
    return enriched


parsed_elements = enrich_images_with_ocr(parsed_elements)

# Test: show OCR results for first image
image_elements = [e for e in parsed_elements if e.modality == "image"]
if image_elements:
    print("Sample OCR result for first image:")
    print(image_elements[0].content[:300])
else:
    print("No images found in this PDF. OCR step skipped.")

## Cell 9 — Chunking Strategy

Split long text blocks into smaller semantic chunks with an **overlapping sliding window**.  
- `chunk_size` controls maximum words per chunk  
- `overlap` words are repeated between consecutive chunks to preserve context at boundaries  
- Table and image elements are treated as atomic chunks (no splitting)

In [ ]:
CHUNK_SIZE = 150   # words per chunk
OVERLAP = 30       # word overlap between chunks


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> List[str]:
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks


def create_chunk_elements(elements: List[ParsedElement]) -> List[ParsedElement]:
    chunked = []
    for el in elements:
        if el.modality == "text":
            for chunk_text_part in chunk_text(el.content):
                chunked.append(ParsedElement(
                    modality="text",
                    content=chunk_text_part,
                    page_number=el.page_number,
                    bounding_box=el.bounding_box
                ))
        else:
            chunked.append(el)
    return chunked


chunked_elements = create_chunk_elements(parsed_elements)

# Save chunks to JSON for inspection
chunks_file = BASE_DIR / "chunks" / f"{doc_id}_chunks.json"
with open(chunks_file, "w", encoding="utf-8") as f:
    json.dump(
        [{"modality": e.modality, "content": e.content[:200], "page": e.page_number} for e in chunked_elements],
        f, indent=2
    )

print(f"Total chunks after splitting: {len(chunked_elements)}")
text_chunks = [e for e in chunked_elements if e.modality == "text"]
print(f"  Text chunks: {len(text_chunks)}")
print(f"  Image chunks: {len([e for e in chunked_elements if e.modality == 'image'])}")
print(f"  Table chunks: {len([e for e in chunked_elements if e.modality == 'table'])}")
if text_chunks:
    print("\nSample text chunk:")
    print(text_chunks[0].content[:200])

## Cell 10 — Text Embedding Pipeline

Embed all text and table chunks using **`all-MiniLM-L6-v2`** from SentenceTransformers.  
This model produces 384-dimensional vectors and runs efficiently on CPU.

In [ ]:
TEXT_EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
TEXT_EMBED_DIM = 384

text_embedder = SentenceTransformer(TEXT_EMBED_MODEL_NAME)
print(f"Text embedding model loaded: {TEXT_EMBED_MODEL_NAME}")


def embed_texts(texts: List[str], batch_size: int = 32) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embeddings = text_embedder.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        all_embeddings.append(embeddings)
    return np.vstack(all_embeddings) if all_embeddings else np.array([])


# Embed text and table elements
text_table_elements = [e for e in chunked_elements if e.modality in ("text", "table")]
text_contents = [e.content for e in text_table_elements]

print(f"Embedding {len(text_contents)} text/table chunks...")
text_embeddings = embed_texts(text_contents)

# Cache to disk
np.save(str(BASE_DIR / "embeddings" / f"{doc_id}_text_embeddings.npy"), text_embeddings)

print(f"Text embedding matrix shape: {text_embeddings.shape}")
print(f"Sample vector (first 5 dims): {text_embeddings[0][:5]}")

## Cell 11 — Image Embedding Pipeline (CLIP)

Embed extracted images using **OpenAI CLIP** (`ViT-B/32` via open_clip).  
CLIP produces 512-dimensional vectors that encode visual semantics — enabling image retrieval by text queries.

This is the **Pro Feature** of VisionRAG: hybrid retrieval using both text and CLIP embeddings.

In [ ]:
CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"
IMAGE_EMBED_DIM = 512

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED, device=device
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
clip_model.eval()
print(f"CLIP model loaded on {device}: {CLIP_MODEL_NAME}")


def embed_image_pil(pil_img: Image.Image) -> np.ndarray:
    img_tensor = clip_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = clip_model.encode_image(img_tensor)
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)
    return embedding.cpu().numpy()[0]


def embed_text_with_clip(text: str) -> np.ndarray:
    """Embed a text query using CLIP's text encoder for image retrieval."""
    tokens = clip_tokenizer([text]).to(device)
    with torch.no_grad():
        embedding = clip_model.encode_text(tokens)
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)
    return embedding.cpu().numpy()[0]


image_elements = [e for e in chunked_elements if e.modality == "image"]
print(f"Embedding {len(image_elements)} images with CLIP...")

image_embeddings = []
for el in image_elements:
    img_path_raw = el.content.split("[IMAGE_PATH: ")[-1].split("]")[0] if "[IMAGE_PATH:" in el.content else el.content
    if Path(img_path_raw).exists():
        pil_img = Image.open(img_path_raw).convert("RGB")
        emb = embed_image_pil(pil_img)
    else:
        emb = np.zeros(IMAGE_EMBED_DIM)
    image_embeddings.append(emb)

image_embeddings = np.array(image_embeddings) if image_embeddings else np.zeros((0, IMAGE_EMBED_DIM))
np.save(str(BASE_DIR / "embeddings" / f"{doc_id}_image_embeddings.npy"), image_embeddings)

print(f"Image embedding matrix shape: {image_embeddings.shape}")

## Cell 12 — Setup Qdrant Vector Database

Initialize a **local Qdrant instance** with two collections:
- `text_chunks` — 384-dim text and table embeddings
- `image_chunks` — 512-dim CLIP image embeddings

Each point stores metadata as a payload: `chunk_id`, `doc_id`, `modality`, `page_number`, `bounding_box`.

In [ ]:
QDRANT_TEXT_COLLECTION = "text_chunks"
QDRANT_IMAGE_COLLECTION = "image_chunks"

qdrant_client = QdrantClient(path=str(BASE_DIR / "qdrant_storage"))


def create_collection_if_not_exists(collection_name: str, vector_dim: int):
    existing = [c.name for c in qdrant_client.get_collections().collections]
    if collection_name not in existing:
        qdrant_client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE)
        )
        print(f"Created collection: {collection_name} (dim={vector_dim})")
    else:
        print(f"Collection already exists: {collection_name}")


create_collection_if_not_exists(QDRANT_TEXT_COLLECTION, TEXT_EMBED_DIM)
create_collection_if_not_exists(QDRANT_IMAGE_COLLECTION, IMAGE_EMBED_DIM)

# Verify
collections = qdrant_client.get_collections().collections
print("\nQdrant collections:")
for c in collections:
    info = qdrant_client.get_collection(c.name)
    print(f"  {c.name}: {info.points_count} points")

## Cell 13 — Store Embeddings in Qdrant

Insert all embeddings as Qdrant `PointStruct` objects.  
Each point carries a rich payload so we can reconstruct context during retrieval.

In [ ]:
def upsert_text_embeddings(
    elements: List[ParsedElement],
    embeddings: np.ndarray,
    doc_id: str
):
    points = []
    for idx, (el, emb) in enumerate(zip(elements, embeddings)):
        vector_id = str(uuid.uuid4())
        chunk_id = register_chunk(
            doc_id=doc_id,
            modality=el.modality,
            text=el.content[:500],
            page_number=el.page_number,
            bounding_box=el.bounding_box,
            vector_id=vector_id
        )
        points.append(PointStruct(
            id=str(uuid.uuid4()),
            vector=emb.tolist(),
            payload={
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "modality": el.modality,
                "page_number": el.page_number,
                "bounding_box": el.bounding_box,
                "text_preview": el.content[:300]
            }
        ))
    if points:
        qdrant_client.upsert(collection_name=QDRANT_TEXT_COLLECTION, points=points)
    return len(points)


def upsert_image_embeddings(
    elements: List[ParsedElement],
    embeddings: np.ndarray,
    doc_id: str
):
    points = []
    for idx, (el, emb) in enumerate(zip(elements, embeddings)):
        if emb.sum() == 0:
            continue
        vector_id = str(uuid.uuid4())
        img_path = el.content.split("[IMAGE_PATH: ")[-1].split("]")[0] if "[IMAGE_PATH:" in el.content else el.content
        ocr_text = el.content.split("[OCR_TEXT: ")[-1].rstrip("]") if "[OCR_TEXT:" in el.content else ""
        chunk_id = register_chunk(
            doc_id=doc_id,
            modality="image",
            text=ocr_text[:500] if ocr_text else None,
            page_number=el.page_number,
            vector_id=vector_id
        )
        points.append(PointStruct(
            id=str(uuid.uuid4()),
            vector=emb.tolist(),
            payload={
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "modality": "image",
                "page_number": el.page_number,
                "image_path": img_path,
                "ocr_text": ocr_text[:300]
            }
        ))
    if points:
        qdrant_client.upsert(collection_name=QDRANT_IMAGE_COLLECTION, points=points)
    return len(points)


n_text = upsert_text_embeddings(text_table_elements, text_embeddings, doc_id)
n_image = upsert_image_embeddings(image_elements, image_embeddings, doc_id)

print(f"Stored {n_text} text/table vectors in '{QDRANT_TEXT_COLLECTION}'")
print(f"Stored {n_image} image vectors in '{QDRANT_IMAGE_COLLECTION}'")

# Verify counts
for col in [QDRANT_TEXT_COLLECTION, QDRANT_IMAGE_COLLECTION]:
    info = qdrant_client.get_collection(col)
    print(f"  {col}: {info.points_count} total points")

## Cell 14 — Hybrid Retrieval System

Implement two retrieval functions:
1. **Text retrieval** — encode query with SentenceTransformer, search `text_chunks`
2. **Image retrieval** — encode query with CLIP text encoder, search `image_chunks`

Both return scored results with payloads for downstream reranking.

In [ ]:
def retrieve_text(
    query: str,
    top_k: int = 5,
    modality_filter: Optional[str] = None
) -> List[Dict]:
    query_vec = text_embedder.encode([query], normalize_embeddings=True)[0].tolist()

    search_filter = None
    if modality_filter:
        search_filter = Filter(
            must=[FieldCondition(key="modality", match=MatchValue(value=modality_filter))]
        )

    results = qdrant_client.search(
        collection_name=QDRANT_TEXT_COLLECTION,
        query_vector=query_vec,
        limit=top_k,
        query_filter=search_filter,
        with_payload=True
    )
    return [
        {"score": r.score, "payload": r.payload, "source": "text"}
        for r in results
    ]


def retrieve_images(
    query: str,
    top_k: int = 3
) -> List[Dict]:
    query_vec = embed_text_with_clip(query).tolist()
    results = qdrant_client.search(
        collection_name=QDRANT_IMAGE_COLLECTION,
        query_vector=query_vec,
        limit=top_k,
        with_payload=True
    )
    return [
        {"score": r.score, "payload": r.payload, "source": "clip"}
        for r in results
    ]


# Test retrieval
test_query = "What is the total revenue for 2024?"
text_results = retrieve_text(test_query, top_k=3)
image_results = retrieve_images(test_query, top_k=2)

print(f"Query: {test_query}")
print(f"\nTop text results:")
for r in text_results:
    print(f"  [score={r['score']:.4f}] page={r['payload'].get('page_number')} | {r['payload'].get('text_preview', '')[:100]}")

print(f"\nTop image results:")
for r in image_results:
    print(f"  [score={r['score']:.4f}] page={r['payload'].get('page_number')} | OCR: {r['payload'].get('ocr_text', '')[:80]}")

## Cell 15 — Cross-Modal Reranker

Combine text and image retrieval results into a single ranked list.  
Strategy: **score fusion with modality weighting** — text results score at full weight, image results at a configurable weight (default 0.7) since their scores come from a different embedding space.

In [ ]:
def rerank_results(
    text_results: List[Dict],
    image_results: List[Dict],
    image_weight: float = 0.7,
    top_k: int = 5
) -> List[Dict]:
    combined = []

    for r in text_results:
        combined.append({**r, "rerank_score": r["score"]})

    for r in image_results:
        combined.append({**r, "rerank_score": r["score"] * image_weight})

    combined.sort(key=lambda x: x["rerank_score"], reverse=True)
    return combined[:top_k]


def format_context_for_llm(reranked_results: List[Dict]) -> str:
    context_parts = []
    for i, r in enumerate(reranked_results):
        payload = r["payload"]
        modality = payload.get("modality", "unknown")
        page = payload.get("page_number", "?")
        score = r["rerank_score"]

        if modality == "image":
            ocr_text = payload.get("ocr_text", "")
            context_parts.append(
                f"[Context {i+1} | IMAGE | page={page} | score={score:.3f}]\n"
                f"OCR text from image: {ocr_text}\n"
            )
        else:
            text_preview = payload.get("text_preview", "")
            context_parts.append(
                f"[Context {i+1} | {modality.upper()} | page={page} | score={score:.3f}]\n"
                f"{text_preview}\n"
            )
    return "\n".join(context_parts)


# Test reranking
reranked = rerank_results(text_results, image_results, image_weight=0.7, top_k=5)
print(f"Reranked top {len(reranked)} results:")
for r in reranked:
    print(f"  [{r['payload'].get('modality')}] rerank_score={r['rerank_score']:.4f} page={r['payload'].get('page_number')}")

context_str = format_context_for_llm(reranked)
print("\nFormatted context (first 400 chars):")
print(context_str[:400])

## Cell 16 — RAG Reasoning with Gemini

Send the retrieved context (text + OCR from images) to Gemini with a structured prompt.  
If the top result is an image, we also pass the PIL image directly for visual reasoning.

In [ ]:
RAG_SYSTEM_PROMPT = """You are a document analysis assistant. You are given retrieved context from a PDF document.
The context may include text passages, extracted table data, and OCR text from images or charts.
Answer the question based only on the provided context.
If the answer is not in the context, say: 'The retrieved context does not contain enough information to answer this question.'
Always cite the page number if available."""


def build_rag_prompt(question: str, context: str) -> str:
    return f"""{RAG_SYSTEM_PROMPT}

RETRIEVED CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""


def get_top_image_if_available(reranked_results: List[Dict]) -> Optional[Image.Image]:
    """Return PIL image of top-ranked image result for visual reasoning."""
    for r in reranked_results:
        if r["payload"].get("modality") == "image":
            img_path = r["payload"].get("image_path", "")
            if img_path and Path(img_path).exists():
                return Image.open(img_path).convert("RGB")
    return None


def rag_answer(question: str, reranked_results: List[Dict]) -> str:
    context = format_context_for_llm(reranked_results)
    prompt = build_rag_prompt(question, context)
    top_image = get_top_image_if_available(reranked_results)
    return call_gemini(prompt, image_pil=top_image)


# Test RAG
answer = rag_answer(test_query, reranked)
print(f"Question: {test_query}")
print(f"Answer: {answer}")

## Cell 17 — End-to-End Question Answering Pipeline

Assemble all stages into a single `query_pipeline` function that takes a question and returns a complete answer with provenance.

In [ ]:
def query_pipeline(
    question: str,
    text_top_k: int = 5,
    image_top_k: int = 3,
    image_weight: float = 0.7
) -> Dict[str, Any]:
    """
    Full VisionRAG pipeline:
      question -> retrieve (text + image) -> rerank -> LLM answer
    """
    logger.info(f"Running query pipeline for: '{question}'")

    text_results = retrieve_text(question, top_k=text_top_k)
    image_results = retrieve_images(question, top_k=image_top_k)
    reranked = rerank_results(text_results, image_results, image_weight=image_weight)

    answer = rag_answer(question, reranked)

    sources = [
        {
            "modality": r["payload"].get("modality"),
            "page_number": r["payload"].get("page_number"),
            "score": round(r["rerank_score"], 4)
        }
        for r in reranked
    ]

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "num_text_results": len(text_results),
        "num_image_results": len(image_results)
    }


# Quick pipeline test
result = query_pipeline("What is the operating margin improvement?")
print("Question:", result["question"])
print("Answer:", result["answer"])
print("Sources:")
for s in result["sources"]:
    print(f"  {s}")

## Cell 18 — Evaluation Tests

Run a set of representative queries to evaluate the system across modalities.  
Results are printed with answer and source attribution.

In [ ]:
evaluation_queries = [
    "What is the total revenue mentioned in the report?",
    "What does the chart on the revenue page show?",
    "What is the year-over-year change in net income?",
    "How many employees does the company have?",
    "What product generates the most revenue?",
]

print("=" * 70)
print("VisionRAG Evaluation Results")
print("=" * 70)

for i, query in enumerate(evaluation_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("-" * 60)
    result = query_pipeline(query)
    print(f"Answer: {result['answer']}")
    top_source = result["sources"][0] if result["sources"] else {}
    print(f"Top source: modality={top_source.get('modality')} page={top_source.get('page_number')} score={top_source.get('score')}")
    print()

## Cell 19 — FastAPI Endpoint Example

A minimal production-ready API endpoint.  
In a real deployment this would be a standalone `main.py` served with `uvicorn main:app --host 0.0.0.0 --port 8000`.

The `/query` endpoint accepts a JSON body with `question` and optional tuning parameters.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

app = FastAPI(title="VisionRAG API", version="1.0.0")


class QueryRequest(BaseModel):
    question: str
    text_top_k: int = 5
    image_top_k: int = 3
    image_weight: float = 0.7


class QueryResponse(BaseModel):
    question: str
    answer: str
    sources: list
    num_text_results: int
    num_image_results: int


@app.get("/health")
def health_check():
    return {"status": "ok", "model": GEMINI_MODEL_NAME}


@app.post("/query", response_model=QueryResponse)
def query_endpoint(request: QueryRequest):
    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty.")
    result = query_pipeline(
        question=request.question,
        text_top_k=request.text_top_k,
        image_top_k=request.image_top_k,
        image_weight=request.image_weight
    )
    return QueryResponse(**result)


# Launch in background thread so notebook stays interactive
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8765, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("FastAPI server started at http://localhost:8765")
print("Docs: http://localhost:8765/docs")

# Test with a direct HTTP call
import time, requests
time.sleep(2)
try:
    resp = requests.post(
        "http://localhost:8765/query",
        json={"question": "What is the gross margin in 2024?"}
    )
    data = resp.json()
    print("API test response:")
    print("  Answer:", data.get("answer", "")[:200])
except Exception as e:
    print(f"API test skipped (server may need a moment): {e}")

## Cell 20 — Performance Improvements

Discussion of key optimizations to move from notebook prototype to production scale.

In [ ]:
performance_guide = """
VISIONRAG PERFORMANCE IMPROVEMENTS
===================================

1. BATCH EMBEDDING
------------------
Current: Each chunk is embedded individually.
Better : Collect all texts/images first, embed in batches of 64-256.
         SentenceTransformer.encode(texts, batch_size=64, show_progress_bar=True)
Benefit: 5-10x throughput improvement on GPU.

2. VECTOR INDEXING (HNSWLIB / Qdrant HNSW)
------------------------------------------
Current: Qdrant local with default HNSW settings.
Better : Tune HNSW parameters for your dataset size:
         VectorParams(size=384, distance=Distance.COSINE,
                      hnsw_config=HnswConfigDiff(m=16, ef_construct=200))
Benefit: Faster ANN search with controlled recall tradeoff.

3. CACHING
----------
Layer 1: Cache query embeddings in Redis (TTL 1 hour).
         If same question asked again, skip embedding.
Layer 2: Cache full RAG answers for frequent queries.
         Use a hash of the normalized question as cache key.
Benefit: Near-zero latency for repeated queries.

4. ASYNC PROCESSING
-------------------
Current: Synchronous pipeline.
Better : Use asyncio + background workers for document ingestion.
         FastAPI endpoint returns a job_id immediately;
         a Celery/RQ worker processes the PDF asynchronously.

5. QUANTIZED EMBEDDINGS
-----------------------
Use int8 vector quantization in Qdrant to reduce storage by 4x.
         qdrant_client.create_collection(...,
             quantization_config=ScalarQuantization(
                 scalar=ScalarQuantizationConfig(type=ScalarType.INT8)))

6. PRODUCTION DOCUMENT PARSING
------------------------------
Replace the PyMuPDF heuristic table detector with:
  - unstructured.io   : best for mixed-layout PDFs
  - camelot           : best for table-heavy PDFs
  - azure-form-recognizer : best for scanned/handwritten docs
"""
print(performance_guide)

## Cell 21 — Production Deployment Guide

### 1. Convert this notebook into a production system

Refactor cells into a Python package:
```
visionrag/
  __init__.py
  config.py          # API keys, model names, constants
  db.py              # SQLite/Postgres schema
  parser.py          # PDF parsing (Cell 7)
  ocr.py             # PaddleOCR (Cell 8)
  chunker.py         # Chunking (Cell 9)
  embedder.py        # Text + CLIP embeddings (Cells 10-11)
  vector_store.py    # Qdrant operations (Cells 12-13)
  retriever.py       # Retrieval + reranking (Cells 14-15)
  rag.py             # Gemini reasoning (Cell 16)
  pipeline.py        # Assembled pipeline (Cell 17)
  api/
    main.py          # FastAPI app (Cell 19)
    schemas.py       # Pydantic models
```

---

### 2. Docker deployment

```dockerfile
# Dockerfile
FROM python:3.10-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY visionrag/ ./visionrag/
COPY visionrag/api/main.py .

ENV GEMINI_API_KEY=your_key_here
EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

```yaml
# docker-compose.yml
version: "3.9"
services:
  visionrag-api:
    build: .
    ports: ["8000:8000"]
    volumes:
      - ./visionrag_data:/app/visionrag_data
    environment:
      - GEMINI_API_KEY=${GEMINI_API_KEY}

  qdrant:
    image: qdrant/qdrant:latest
    ports: ["6333:6333"]
    volumes:
      - ./qdrant_storage:/qdrant/storage
```

---

### 3. Next.js frontend

```
npx create-next-app visionrag-ui
```

Key pages:
- `/upload` — drag-and-drop PDF upload calling `POST /ingest`
- `/chat` — chat interface calling `POST /query` with streaming
- `/sources` — show retrieved page thumbnails alongside each answer

Connect to API:
```javascript
const res = await fetch("http://localhost:8000/query", {
  method: "POST",
  headers: { "Content-Type": "application/json" },
  body: JSON.stringify({ question: userInput })
});
const data = await res.json();
// data.answer, data.sources
```

---

### 4. Portfolio improvements

| Improvement | Impact |
|-------------|--------|
| Add a streaming `/query/stream` SSE endpoint | Feels faster, modern UX |
| Persist Qdrant to a managed cloud instance (Qdrant Cloud free tier) | No local storage required |
| Add a document library page showing all indexed PDFs | Demonstrates full system |
| Implement a BM25 sparse retrieval layer for hybrid search | Shows advanced IR knowledge |
| Add evaluation metrics (MRR, NDCG) using RAGAS library | Shows ML rigor |
| Deploy backend on Railway.app and frontend on Vercel | Fully public live demo |
| Record a 2-minute Loom demo | Greatly increases recruiter engagement |
